In [9]:
import numpy as np
import pandas as pd
from os.path import join

In [25]:
def assign_ate_groups(intervention: str, variable: str) -> tuple[str, str]:
    """
    Assign distribution and structure groups to an (intervention, variable) pair.
    
    Args:
        intervention: String like "S1=0.5", "S3=1.0", etc.
        variable: String like "X1", "X2", etc.
    
    Returns:
        tuple: (distribution_group, structure_group)
            - distribution_group: "ID" (in-distribution) or "OOD" (out-of-distribution)
            - structure_group: "zero" (no causal path, expect ATE=0) or "nonzero" (has causal path)
    """
    # Parse intervention
    src_var, value_str = intervention.split("=")
    value = float(value_str)
    
    # Distribution classification (ID vs OOD)
    # OOD values as defined in ate_ground_truth.json
    ood_interventions = {
        "S3": [1.0],
        "S5": [2.5],
    }
    
    is_ood = src_var in ood_interventions and value in ood_interventions[src_var]
    dist_group = "OOD" if is_ood else "ID"
    
    # Structure classification (zero vs nonzero expected ATE)
    # Based on SCM DAG: S → X relationships
    descendants = {
        "S1": [],                       # Dangling, no effect on any X
        "S2": ["X1", "X5"],             # S2 → X1 → X5
        "S3": ["X2", "X3", "X4", "X5"], # S3 → X2,X3; X2 → X4,X5
        "S4": ["X4"],                   # S4 → X4
        "S5": ["X4"],                   # S5 → X4
    }
    
    has_effect = variable in descendants.get(src_var, [])
    struct_group = "nonzero" if has_effect else "zero"
    
    return dist_group, struct_group


def add_ate_groups(df: pd.DataFrame) -> pd.DataFrame:
    """Add distribution and structure group columns to dataframe."""
    df = df.copy()
    groups = df.apply(lambda row: assign_ate_groups(row["intervention"], row["variable"]), axis=1)
    df["dist_group"] = [g[0] for g in groups]
    df["struct_group"] = [g[1] for g in groups]
    return df

# Usage:
# df = add_ate_groups(df)
# summary = df.groupby(["dist_group", "struct_group", "hard"])["abs_error_mean"].agg(["mean", "std", "count"])
# print(summary.to_latex())



def aggregate_ate_by_groups(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate ATE errors by (model, dataset, hard) for each of the 4 structure/distribution groups.
    
    Uses uncertainty propagation: when averaging N values with uncertainties σ_i,
    the combined uncertainty is sqrt(Σ σ_i² / N²) for independent measurements.
    
    Args:
        df: DataFrame with columns: model, dataset, hard, dist_group, struct_group,
            abs_error_mean, abs_error_std, rel_error_mean, rel_error_std
    
    Returns:
        DataFrame with aggregated metrics per (model, dataset, hard) and structure/distribution group
    """
    results = []
    
    # Group by experiment configuration
    for (model, dataset, hard), exp_df in df.groupby(["model", "dataset", "hard"]):
        
        # For each of the 4 groups
        for dist in ["ID", "OOD"]:
            for struct in ["zero", "nonzero"]:
                mask = (exp_df["dist_group"] == dist) & (exp_df["struct_group"] == struct)
                group_df = exp_df[mask]
                
                if len(group_df) == 0:
                    continue
                
                n = len(group_df)
                
                if "abs_error_std" in group_df.columns:
                    # Absolute error aggregation with uncertainty propagation
                    abs_means = group_df["abs_error_mean"].values
                    abs_stds = group_df["abs_error_std"].fillna(0).values

                    agg_abs_mean = np.mean(abs_means)
                    # Combined std: sqrt(sum of variances) / N for averaging
                    agg_abs_std = np.sqrt(np.sum(abs_stds**2)) / n

                    # Relative error aggregation (drop NaN for zero ATE cases)
                    scaled_df = group_df.dropna(subset=["scaled_error_mean"])
                    if len(scaled_df) > 0:
                        n_rel = len(scaled_df)
                        scaled_means = scaled_df["scaled_error_mean"].values
                        scaled_stds = scaled_df["scaled_error_std"].fillna(0).values

                        agg_scaled_mean = np.mean(scaled_means)
                        agg_scaled_std = np.sqrt(np.sum(scaled_stds**2)) / n_rel
                    else:
                        agg_scaled_mean = np.nan
                        agg_scaled_std = np.nan

                    results.append({
                        "model": model,
                        "dataset": dataset,
                        "hard": hard,
                        "dist_group": dist,
                        "struct_group": struct,
                        "group_label": f"{dist}_{struct}",
                        "n_pairs": n,
                        "abs_error": agg_abs_mean,
                        "abs_error_std": agg_abs_std,
                        "scaled_error": agg_scaled_mean,
                        "scaled_error_std": agg_scaled_std,
                    })
                else:
                    abs_means = group_df["abs_error"].values
                    agg_abs_mean = np.mean(abs_means)
                    results.append({
                        "model": model,
                        "dataset": dataset,
                        "hard": hard,
                        "dist_group": dist,
                        "struct_group": struct,
                        "group_label": f"{dist}_{struct}",
                        "n_pairs": 0,
                        "abs_error": agg_abs_mean,
                        "abs_error_std": None,
                        "scaled_error": None,
                        "scaled_error_std": None,
                    })
    
    return pd.DataFrame(results)


def format_latex_table(agg_df: pd.DataFrame) -> str:
    """
    Format aggregated results as LaTeX table.
    
    Rows: (model, dataset, hard)
    Columns: 4 groups × 2 metrics = 8 columns
    """
    # Pivot to wide format
    pivot_df = agg_df.pivot_table(
        index=["model", "dataset", "hard"],
        columns="group_label",
        values=["abs_error", "abs_error_std", "rel_error", "rel_error_std"],
        aggfunc="first"
    )
    
    # Format as mean ± std
    def fmt(mean, std):
        if pd.isna(mean):
            return "-"
        if pd.isna(std) or std == 0:
            return f"{mean:.3f}"
        return f"{mean:.3f} ± {std:.3f}"
    
    return pivot_df


# Usage:
# df = add_ate_groups(df)  # First add groups
# agg_df = aggregate_ate_by_groups(df)
# 
# # Display as pivot table
# display(agg_df.pivot_table(
#     index=["model", "dataset", "hard"],
#     columns="group_label",
#     values=["abs_error", "rel_error"]
# ))
#
# # Or export to LaTeX
# print(agg_df.to_latex(index=False, float_format="%.3f"))


In [29]:
ate_summary_filename = "ate_summary.csv"
experiment_summary_filename = "experiment_summary.csv"
intermediate_path = "eval/eval_seed_sweep/files"

baseline_list = [
    {
        "model"     : "baseline",
        "dataset"   : "scm1",
        "hard"      : False,
        "filepath"  : "../experiments/baseline/euler/vanilla_transformer_scm1_62011281"
        },
    {
        "model"     : "baseline",
        "dataset"   : "scm1",
        "hard"      : True,
        "filepath"  : "../experiments/baseline/euler/vanilla_transformer_scm1_hard_61995166"
        },
    {
        "model"     : "baseline",
        "dataset"   : "scm2",
        "hard"      : False,
        "filepath"  : "../experiments/baseline/euler/vanilla_transformer_scm2_62022298"
        },
    {
        "model"     : "baseline",
        "dataset"   : "scm2",
        "hard"      : True,
        "filepath"  : "../experiments/baseline/euler/vanilla_transformer_scm2_hard_62022335"
        },
    {
        "model"     : "baseline",
        "dataset"   : "scm3",
        "hard"      : False,
        "filepath"  : "../experiments/baseline/euler/vanilla_transformer_scm3_62022405"
        },
    {
        "model"     : "baseline",
        "dataset"   : "scm3",
        "hard"      : True,
        "filepath"  : "../experiments/baseline/euler/vanilla_transformer_scm3_hard_62022367"
        },
]

df = None

for d in baseline_list:    
    df_ = pd.read_csv(join(d["filepath"], intermediate_path, ate_summary_filename))
    df_["model"] = d["model"]
    df_["dataset"] = d["dataset"]
    df_["hard"] = d["hard"]
    
    if df is None:
        df = df_
    else:
        df = pd.concat([df, df_], axis=0)
        
df = add_ate_groups(df)
agg_df = aggregate_ate_by_groups(df)

display(agg_df.pivot_table(
    index=["model", "dataset", "hard"],
    columns="group_label",
    values=["abs_error", "abs_error_std"]
))

abs_error                                  \
group_label            ID_nonzero   ID_zero OOD_nonzero  OOD_zero   
model    dataset hard                                               
baseline scm1    False   0.373916  0.044688    0.456710  1.034554   
                 True    0.375064  0.000000    0.430294  0.000000   
         scm2    False   1.538356  0.002954    0.693298  0.007034   
                 True    1.575754  0.000000    0.666868  0.000000   
         scm3    False   1.553633  0.003841    0.688975  0.011567   
                 True    1.568560  0.000000    0.660365  0.000000   

                       abs_error_std                                  
group_label               ID_nonzero   ID_zero OOD_nonzero  OOD_zero  
model    dataset hard                                                 
baseline scm1    False      0.008130  0.001981    0.027280  0.026265  
                 True       0.011330  0.000000    0.051808  0.000000  
         scm2    False      0.028353  0.000853    0.037810  0.003308  
                 True       0.038140  0.000000    0.037340  0.000000  
         scm3    False      0.027331  0.001580    0.036858  0.006059  
                 True       0.041935  0.000000    0.030072  0.000000

In [30]:



df_test = pd.read_csv("../experiments/noise_aware_single/scm1/euler/na_single_Toeplitz_CC_scm1_62079347/eval/eval_ate/files/ate_metrics.csv")
df_test["model"] = "causaliT"
df_test["dataset"] = "scm1"
df_test["hard"] = False

ate_summary_filename = "ate_metrics.csv"
intermediate_path = "eval/eval_ate/files"

model_list = [
    {
        "model"     : "causaliT_CC",
        "dataset"   : "scm1",
        "hard"      : False,
        "filepath"  : "../experiments/noise_aware_single/scm1/euler/na_single_Toeplitz_CC_scm1_62079347"
        },
    {
        "model"     : "causaliT_SM",
        "dataset"   : "scm1",
        "hard"      : False,
        "filepath"  : "../experiments/noise_aware_single/scm1/euler/na_single_Toeplitz_SM_scm1_62078885"
        },
    {
        "model"     : "causaliT_CC",
        "dataset"   : "scm2",
        "hard"      : False,
        "filepath"  : "../experiments/noise_aware_single/scm2/euler/na_single_Toeplitz_CC_scm2_62079844"
        },
    {
        "model"     : "causaliT_SM",
        "dataset"   : "scm2",
        "hard"      : False,
        "filepath"  : "../experiments/noise_aware_single/scm2/euler/na_single_Toeplitz_SM_scm2_62079876"
        },
    {
        "model"     : "causaliT_CC",
        "dataset"   : "scm3",
        "hard"      : False,
        "filepath"  : "../experiments/noise_aware_single/scm3/na_single_Toeplitz_CC_scm3_62079934"
        },
    {
        "model"     : "causaliT_SM",
        "dataset"   : "scm3",
        "hard"      : False,
        "filepath"  : "../experiments/noise_aware_single/scm3/na_single_Toeplitz_SM_scm3_62079922"
        },
    
]


df_model = None
for d in model_list:    
    df_ = pd.read_csv(join(d["filepath"], intermediate_path, ate_summary_filename))
    df_["model"] = d["model"]
    df_["dataset"] = d["dataset"]
    df_["hard"] = d["hard"]
    
    if df_model is None:
        df_model = df_
    else:
        df_model = pd.concat([df_model, df_], axis=0)



df_model = add_ate_groups(df_model)
agg_df_model = aggregate_ate_by_groups(df_model)



display(agg_df.pivot_table(
    index=["model", "dataset", "hard"],
    columns="group_label",
    values=["abs_error", "abs_error_std"]
))

display(agg_df_model.pivot_table(
    index=["model", "dataset", "hard"],
    columns="group_label",
    values=["abs_error"]
))


abs_error                                  \
group_label            ID_nonzero   ID_zero OOD_nonzero  OOD_zero   
model    dataset hard                                               
baseline scm1    False   0.373916  0.044688    0.456710  1.034554   
                 True    0.375064  0.000000    0.430294  0.000000   
         scm2    False   1.538356  0.002954    0.693298  0.007034   
                 True    1.575754  0.000000    0.666868  0.000000   
         scm3    False   1.553633  0.003841    0.688975  0.011567   
                 True    1.568560  0.000000    0.660365  0.000000   

                       abs_error_std                                  
group_label               ID_nonzero   ID_zero OOD_nonzero  OOD_zero  
model    dataset hard                                                 
baseline scm1    False      0.008130  0.001981    0.027280  0.026265  
                 True       0.011330  0.000000    0.051808  0.000000  
         scm2    False      0.028353  0.000853    0.037810  0.003308  
                 True       0.038140  0.000000    0.037340  0.000000  
         scm3    False      0.027331  0.001580    0.036858  0.006059  
                 True       0.041935  0.000000    0.030072  0.000000

abs_error                                
group_label               ID_nonzero   ID_zero OOD_nonzero  OOD_zero
model       dataset hard                                            
causaliT_CC scm1    False   0.384611  0.016060    0.157352  0.348317
            scm2    False   1.625300  0.014592    0.691049  0.027631
            scm3    False   1.674233  0.011050    0.593091  0.043596
causaliT_SM scm1    False   0.312324  0.020461    0.460228  0.000511
            scm2    False   1.709328  0.013953    0.603218  0.075372
            scm3    False   1.622594  0.007729    0.495814  0.093337

In [ ]:
df_model